In [ ]:
import pandas as pd
import numpy as np
import hashlib, textwrap
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

FILES = ["general_ledger.csv", "monthly_financials.csv", "accounts_receivable_aging.csv", "management_adjustments.csv"]
h = lambda: [hashlib.md5(open(f, "rb").read()).hexdigest() for f in FILES]
H = h()
g = pd.read_csv(FILES[0], parse_dates=["Transaction_Date", "Posting_Date"])
m = pd.read_csv(FILES[1])
ar = pd.read_csv(FILES[2])
a = pd.read_csv(FILES[3])
m["Year"] = m.Month.str[:4].astype(int)
g["Year"] = g.Posting_Date.dt.year
g["Amount"] = (g.Debit - g.Credit).abs()
COA = {4010: "Revenue", 5010: "COGS", 6010: "Payroll_Expenses", 6020: "Rent", 6030: "SGA_Expenses", 6040: "Depreciation_Amortization", 6100: "Restructuring"}

def report(path="due_diligence_findings_report.pdf"):
    f = "${:,.0f}".format
    wr = lambda s: textwrap.fill(s, 104, initial_indent="  ", subsequent_indent="     ")
    why = {"ADJ-001": "LEGAL-SET-2024 not in GL; no matching SG&A spike in 2024",
           "ADJ-002": "Restructuring repeats in Feb-2025: not one-time",
           "ADJ-003": "Restructuring also charged in Nov-2024: recurring",
           "ADJ-004": "Pro forma: needs CEO contract and $280k market benchmark",
           "ADJ-005": "CONS-ERP-2025 not in GL; only consulting is CONS-2025-Q4",
           "ADJ-006": "EBITDA already excludes D&A: add-back double counts",
           "ADJ-007": "Pro forma: needs market lease study; lease must reset at close",
           "ADJ-008": "Duplicate confirmed in GL (PO-2024-0892)",
           "ADJ-009": "FX-REVAL-2025 not in GL; SG&A posted as monthly totals"}
    rs = g[g.Account_Code.eq(6100)]
    rv = ib[ib.Credit.gt(0)]
    rest = w.Outstanding_Amount.drop("IronBridge Works")
    adj_ar = ar.Outstanding_Amount.sum() - F.iloc[4]
    up = E.defensible + b.get("qualify", 0)
    e24 = m.EBITDA[m.Year.eq(2024)].sum()
    L = ["NORTHSTAR INDUSTRIAL COMPONENTS LTD. - FINANCIAL DUE DILIGENCE FINDINGS REPORT",
         "Sources: general_ledger, monthly_financials, accounts_receivable_aging, management_adjustments (Jan-2024 to Dec-2025)", "",
         "1. EXECUTIVE SUMMARY",
         wr(f"Management presents 2025 adjusted EBITDA of {f(E.management)}. The evidence supports {f(E.defensible)}: reported EBITDA of "
            f"{f(E.reported)} less {f(x)} of consulting cost that is in the general ledger but missing from the management accounts. "
            f"Qualified adjustments of {f(b.get('qualify', 0))} would lift it to {f(up)} only if documented. Receivables of "
            f"{f(ar.Outstanding_Amount.sum())} fall to {f(adj_ar)} after a full provision for IronBridge Works, which is in administration. "
            f"The ledger shifts {f(F.iloc[0])} of revenue from 2024 to 2025: a timing difference with no two-year impact."), "",
         "2. KEY FINANCIAL DISCREPANCIES (general ledger by posting year minus management accounts)",
         *rec.T.map(f).to_string().split("\n"), "",
         "3. QUANTIFIED EARNINGS AND WORKING CAPITAL RISKS (each amount counted once)",
         f"  EBITDA 2025: management claim vs defensible          {f(E.management - E.defensible):>12}",
         f"     rejected {f(b.get('reject', 0))} + qualified {f(b.get('qualify', 0))} + unrecorded consulting {f(x)}",
         f"  Receivables: IronBridge full provision                {f(F.iloc[4]):>12}",
         f"  Receivables: over 90 days, evidence insufficient      {f(rest.sum()):>12}   ({', '.join(rest.index)})",
         f"  Revenue timing 2024 to 2025 (no EBITDA effect)        {f(F.iloc[0]):>12}",
         f"  2024 duplicate cost (2024 EBITDA understated)         {f(F.iloc[1]):>12}   no 2025 effect", "",
         "4. MANAGEMENT EBITDA ADJUSTMENTS (accepted / rejected / qualified)",
         *[f"  {i} {r.Period}  {f(r.Amount):>9}  {r.verdict:<8} {why[i]}" for i, r in a.set_index("Adjustment_ID").iterrows()],
         f"  2025: reported {f(E.reported)} | management {f(E.management)} | defensible {f(E.defensible)} | if qualified supported {f(up)}",
         f"  2024: reported {f(e24)} + accepted ADJ-008 {f(A['ADJ-008'])} = {f(e24 + A['ADJ-008'])}", "",
         "5. RECEIVABLES ASSESSMENT (over 90 days)",
         *[f"  {n:<22}{f(r.Outstanding_Amount):>10}  {r.Days_Overdue:>4} days  {r.Collection_Notes}" for n, r in w.iterrows()],
         wr(f"IronBridge was invoiced {f(rv.Credit.sum())} after August 2025 ({', '.join(rv.Document_Reference)}). These invoices are not in "
            "the aging and the ledger records no cash receipts, so collection cannot be confirmed."), "",
         "6. FIVE FINDINGS"]
    info = [
        (f"{c.Document_Reference.iloc[0]}, {c.Counterparty.iloc[0]}: dated {c.Transaction_Date.iloc[0]:%Y-%m-%d}, posted {c.Posting_Date.iloc[0]:%Y-%m-%d}",
         f"Revenue by posting year, GL minus management accounts: 2024 = {f(rec.Revenue[2024])}; 2025 = {f(rec.Revenue[2025])}; two-year net = {f(rec.Revenue.sum())}",
         "GL understates 2024 and overstates 2025 revenue; management accounts are correct (accrual). Cut-off control weakness",
         "Reclassify to 2024; test cut-off at both year ends", "High - confirmed timing difference"),
        (f"{d.Document_Reference.iloc[0]}, {d.Counterparty.iloc[0]}: {' and '.join(d.Transaction_ID)}, posted {', '.join(d.Posting_Date.dt.strftime('%d-%b-%Y'))}",
         f"Total debited {f(d.Debit.sum())}; one posting is valid, so overstatement = {f(F.iloc[1])}",
         "2024 COGS overstated and EBITDA understated; payable overstated if unpaid. Supports ADJ-008 (2024 only)",
         "Obtain TechServ credit note or vendor statement", "High on duplicate; medium on recovery"),
        ("ADJ-005 (CONS-ERP-2025): 0 GL rows. CONS-2025-Q4, Consulting Partners LLC, 2025-12-15",
         f"Add-back rejected {f(A['ADJ-005'])} + GL SG&A 2025 above management accounts by {f(x)}",
         "Management EBITDA overstated by both amounts", "Request ERP advisory invoices; book the missing consulting cost", "High - referenced document absent from the ledger"),
        (f"{', '.join(rs.Document_Reference)}: {f(rs.Debit.iloc[0])} ({rs.Posting_Date.iloc[0]:%b-%Y}) and {f(rs.Debit.iloc[1])} ({rs.Posting_Date.iloc[1]:%b-%Y})",
         f"Restructuring in consecutive years; ADJ-003 add-back rejected = {f(F.iloc[3])} (ADJ-002 rejected for 2024)",
         "Restructuring behaves as a recurring cost of the operating model", "Request 2026 restructuring plan", "Medium-high - recurrence is a judgment"),
        ("INV2024-0845, IronBridge Works (C008): 407 days overdue, administration proceedings, legal counsel engaged 2025-08",
         f"Full provision {f(F.iloc[4])}; adjusted receivables {f(ar.Outstanding_Amount.sum())} - {f(F.iloc[4])} = {f(adj_ar)}",
         f"Working capital overstated; plus {f(rv.Credit.sum())} billed after administration with no evidence of collection",
         "Obtain administrator's report and post-close receipts", "High on provision; post-administration billing unresolved")]
    for (t, amt), (src, calc, imp, act, conf) in zip(F.items(), info):
        L += [f"  {t} - {f(amt)}", wr(f"Records: {src}"), wr(f"Calculation: {calc}"), wr(f"Impact: {imp}"),
              wr(f"Follow-up: {act}"), wr(f"Confidence: {conf}"), ""]
    L += ["7. LIMITATIONS AND RECOMMENDED FOLLOW-UP",
          wr(f"No cash receipts in the ledger: Accounts Receivable only has debits ({f(g.Debit[g.Account_Code.eq(1200)].sum())}) and none of "
             "the 9 aging invoices appear in the ledger, so the aging cannot be reconciled to the books."),
          wr("Payroll and SG&A are posted as single monthly totals, so ADJ-004 and ADJ-009 cannot be tested at transaction level."),
          wr(f"{', '.join(rest.index)}: {f(rest.sum())} over 90 days with no dispute flag; evidence is insufficient to provide or to clear."),
          wr("Open requests: IronBridge administrator's report; ERP advisory invoices; TechServ credit note; 2026 restructuring plan; "
             "CEO contract and pay benchmark; market lease study; FX revaluation schedule; bank statements to tie receivables.")]
    txt = "\n".join(L).split("\n")
    with PdfPages(path) as pdf:
        for i in range(0, len(txt), 80):
            fig = plt.figure(figsize=(8.5, 11))
            fig.text(0.05, 0.96, "\n".join(txt[i:i + 80]), family="monospace", size=7, va="top", parse_math=False)
            pdf.savefig(fig)
            plt.close(fig)
    print("saved", path)

for n, d in zip(FILES, [g, m, ar, a]):
    print(n, d.shape, "| nulls:", d.isna().sum().sum(), "| duplicate rows:", d.duplicated().sum())
print("GL debits - credits:", g.Debit.sum() - g.Credit.sum(), "| duplicate transaction IDs:", g.Transaction_ID.duplicated().sum())
print("AR aging total:", ar.Outstanding_Amount.sum(), "| aging invoices found in GL:", ar.Invoice_Number.isin(g.Document_Reference).sum())
m.groupby("Year")[["Revenue", "EBITDA", "Depreciation_Amortization", "EBIT", "Net_Income"]].sum()

In [ ]:
g.groupby("Account_Name")[["Debit", "Credit"]].sum()

In [ ]:
y = g.groupby(["Year", g.Account_Code.map(COA)]).Amount.sum().unstack()
rec = y - m.groupby("Year")[y.columns].sum()
rec

In [ ]:
c = g[g.Transaction_Date.dt.year < g.Year]
c

In [ ]:
d = g[g.Debit.gt(0) & g.duplicated(["Document_Reference", "Debit"], keep=False)]
d

In [ ]:
a["in_gl"] = a.Supporting_Document_Reference.isin(g.Document_Reference)
a

In [ ]:
g[g.Account_Code.eq(6100) | g.Counterparty.str.contains("Consult")]

In [ ]:
a["verdict"] = np.where(a.Adjustment_ID.str[-1].isin(list("12356")), "reject", np.where(a.in_gl, "accept", "qualify"))
b = a[a.Period.eq(2025)].groupby("verdict").Amount.sum()
e, x = m.EBITDA[m.Year.eq(2025)].sum(), rec.SGA_Expenses[2025]
E = pd.Series([e, e + b.sum(), e - x], ["reported", "management", "defensible"])
E

In [ ]:
w = ar[ar.Days_Overdue.gt(90)].set_index("Customer_Name")
w

In [ ]:
ib = g[g.Counterparty.eq("IronBridge Works")].tail(6)
ib

In [ ]:
A = a.set_index("Adjustment_ID").Amount
F = pd.Series([rec.Revenue[2025], d.Debit.sum() / 2, A["ADJ-005"] + x, A["ADJ-003"], w.Outstanding_Amount["IronBridge Works"]])
F.index = ["F1 cutoff", "F2 duplicate", "F3 consulting", "F4 restructuring", "F5 IronBridge"]
F

In [ ]:
report()
print(rec.Revenue.sum(), d.Debit.sum() - F.iloc[1], a.verdict.notna().all(), h() == H)